# Chapter 2: Data Preprocessing
## 📌 Summary
Preprocessing adalah langkah krusial sebelum melatih model ML. Chapter ini mencakup **scaling, normalization, encoding kategorikal, imputation**, dan **feature extraction**.

## 🧠 Theoretical Deep-Dive

### 2.1 Feature Scaling
- **StandardScaler**: z = (x−μ)/σ → mean=0, std=1. Ideal untuk SVM, KNN, Linear models
- **MinMaxScaler**: x' = (x−min)/(max−min) → range [0,1]. Sensitif outlier
- **RobustScaler**: Gunakan median & IQR → tahan outlier

### 2.2 Normalization
Normalization (L1/L2) beroperasi **per-baris** (per-sample). Berguna untuk text data dan algoritma yang menghitung dot product.

### 2.3 Encoding Categorical Variables
- **OrdinalEncoder**: kategori → integer (hanya jika ada urutan)
- **OneHotEncoder**: kategori → binary columns (no ordinal assumption) → default pilihan
- **LabelEncoder**: untuk target variable saja

### 2.4 Imputation
**SimpleImputer** mengisi NaN dengan mean/median/most_frequent. **KNNImputer** menggunakan tetangga terdekat untuk estimasi lebih akurat.

### 2.5 ColumnTransformer
Memungkinkan aplikasi transformer berbeda ke kolom berbeda dalam satu pipeline.

## 💻 Code Reproduction

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

X = np.array([[1.0], [2.0], [3.0], [4.0], [100.0]])  # 100 = outlier

for name, scaler in [("Standard", StandardScaler()), 
                      ("MinMax", MinMaxScaler()), 
                      ("Robust", RobustScaler())]:
    X_scaled = scaler.fit_transform(X)
    print(f"{name:10}: {X_scaled.flatten().round(3)}")

In [ ]:
from sklearn.preprocessing import Normalizer
import numpy as np

X = np.array([[4, 1, 2, 2],
              [1, 3, 9, 3],
              [5, 7, 5, 1]])

for norm in ["l1", "l2", "max"]:
    result = Normalizer(norm=norm).fit_transform(X)
    print(f"Norm={norm}:", result.round(3))

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
import numpy as np

X_cat = np.array([["Low", "Cat"], ["High", "Dog"], ["Medium", "Cat"], ["High", "Bird"]])

oe = OrdinalEncoder()
print("OrdinalEncoder:\n", oe.fit_transform(X_cat))

ohe = OneHotEncoder(sparse_output=False)
print("\nOneHotEncoder:\n", ohe.fit_transform(X_cat))
print("Feature names:", ohe.get_feature_names_out())

In [ ]:
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer

X = np.array([[1, 2, np.nan],
              [3, np.nan, 5],
              [np.nan, 8, 9],
              [4, 5, 6]])

for strategy in ["mean", "median", "most_frequent"]:
    imp = SimpleImputer(strategy=strategy)
    print(f"Strategy={strategy}:\n", imp.fit_transform(X).round(2))

knn_imp = KNNImputer(n_neighbors=2)
print("KNNImputer:\n", knn_imp.fit_transform(X).round(2))

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
import numpy as np

X = np.array([[2, 3], [4, 5]])

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)
print("Original:", X.shape, "→ Poly:", X_poly.shape)
print("Feature names:", poly.get_feature_names_out(["x1", "x2"]))
print("Transformed:\n", X_poly)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "age": [25.0, np.nan, 35.0, 45.0, 28.0],
    "salary": [50000, 60000, np.nan, 80000, 55000],
    "city": ["Jakarta", "Bandung", "Jakarta", "Surabaya", "Bandung"]
})

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, ["age", "salary"]),
    ("cat", categorical_transformer, ["city"])
])

X_processed = preprocessor.fit_transform(df)
print("Processed shape:", X_processed.shape)
print("Processed data:\n", X_processed.round(3))

## ✅ Chapter Summary
- **StandardScaler** → default untuk model berbasis gradient/distance
- **RobustScaler** → ketika ada outlier signifikan
- **OneHotEncoder** → default untuk fitur kategorikal
- **Pipeline + ColumnTransformer** → cara production-ready untuk preprocessing mixed data
- Selalu `fit` scaler pada **training data saja**, lalu `transform` test data